In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'transformers', 'datasets', 'scikit-learn',
    'matplotlib', 'seaborn', 'pandas', 'numpy',
], check=True)
print('All packages ready.')

In [ ]:
import json, os, re
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split

os.makedirs('IMDB', exist_ok=True)

CONTRACTION_MAP = {
    "ain't": "is not", "aren't": "are not", "can't": "can not",
    "couldn't": "could not", "didn't": "did not", "doesn't": "does not",
    "don't": "do not", "hadn't": "had not", "hasn't": "has not",
    "haven't": "have not", "he'd": "he would", "he'll": "he will",
    "he's": "he is", "i'd": "i would", "i'll": "i will", "i'm": "i am",
    "i've": "i have", "isn't": "is not", "it'd": "it would",
    "it'll": "it will", "it's": "it is", "let's": "let us",
    "might've": "might have", "must've": "must have", "mustn't": "must not",
    "needn't": "need not", "shan't": "shall not", "she'd": "she would",
    "she'll": "she will", "she's": "she is", "should've": "should have",
    "shouldn't": "should not", "that's": "that is", "there's": "there is",
    "they'd": "they would", "they'll": "they will", "they're": "they are",
    "they've": "they have", "wasn't": "was not", "we'd": "we would",
    "we'll": "we will", "we're": "we are", "we've": "we have",
    "weren't": "were not", "what'll": "what will", "what're": "what are",
    "what's": "what is", "what've": "what have", "where's": "where is",
    "who'd": "who would", "who'll": "who will", "who's": "who is",
    "who've": "who have", "won't": "will not", "wouldn't": "would not",
    "you'd": "you would", "you'll": "you will", "you're": "you are",
    "you've": "you have",
}

def preprocess_text(text):
    t = text.lower()
    t = re.sub(r'<br\s*/?>',  ' ', t)
    t = t.replace('"', ' ')
    t = t.replace('\r', ' ').replace('\n', ' ').replace('\t', ' ')
    for c, exp in CONTRACTION_MAP.items():
        t = t.replace(c, exp)
    t = re.sub(r"\b(\w+)'s\b", r'\1', t)
    for ch in ['..', '--', '/', ';', '(', ')']:
        t = t.replace(ch, ' ')
    t = re.sub(r'\s+', ' ', t).strip()
    return t

print('Downloading IMDB ...')
raw = load_dataset('imdb')
train_full = raw['train']
test_data  = raw['test']

indices    = list(range(len(train_full)))
labels_all = [train_full[i]['label'] for i in indices]
tr_idx, vl_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels_all)

splits = {
    'train':      [train_full[i] for i in tr_idx],
    'validation': [train_full[i] for i in vl_idx],
    'test':       [test_data[i]  for i in range(len(test_data))],
}

for split_name, rows in splits.items():
    out_path = os.path.join('IMDB', f'{split_name}.json')
    if os.path.exists(out_path):
        print(f'  {split_name}.json already exists, skipping.')
        continue
    records = []
    for row in rows:
        records.append({
            'text_original':     row['text'],
            'text_preprocessed': preprocess_text(row['text']),
            'label':             row['label'],
        })
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f'  Saved {split_name}.json  ({len(records)} samples)')

print('IMDB data ready.')

In [ ]:
import json, os, random, time, shutil, zipfile
import numpy as np
import pandas as pd
from datetime import datetime

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, RobertaModel
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
from sklearn.manifold import TSNE

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

SEED          = 42
DATASET       = 'IMDB'
MODEL_NAME    = 'roberta-base'
NUM_EPOCHS    = 3
BATCH_SIZE    = 32
LEARNING_RATE = 2e-5
MAX_LENGTH    = 256
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.06
PROJ_DIM      = 128
TEMPERATURE   = 0.3
LAMBDA_SCL    = 0.5
TEXT_COLUMN   = 'text_original'
LABEL_COLUMN  = 'label'
TEST_LABELS   = True
ROOT_DIR      = os.getcwd()

CE_ZIP_PATH = 'results_IMDB_RoBERTa-CE.zip'

set_seed(SEED)
sns.set_style('whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Dataset: {DATASET}  |  Model: {MODEL_NAME}')
print(f'Epochs: {NUM_EPOCHS}  |  BS: {BATCH_SIZE}  |  LR: {LEARNING_RATE}  |  MAX_LEN: {MAX_LENGTH}')
print(f'Temp: {TEMPERATURE}  |  Lambda SCL: {LAMBDA_SCL}')

In [ ]:
dataset_path = os.path.join(ROOT_DIR, DATASET)

def load_split(split_name):
    fp = os.path.join(dataset_path, f'{split_name}.json')
    if not os.path.exists(fp):
        raise FileNotFoundError(f'{fp} not found.')
    with open(fp, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f'  {split_name}: {len(df)} samples')
    return df

print(f'Loading data from: {dataset_path}')
df_train      = load_split('train')
df_validation = load_split('validation')
df_test       = load_split('test')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IMDBDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc  = tokenizer(self.texts[idx], max_length=MAX_LENGTH,
                         padding='max_length', truncation=True, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_texts  = df_train[TEXT_COLUMN].fillna('').tolist()
train_labels = df_train[LABEL_COLUMN].tolist()
val_texts    = df_validation[TEXT_COLUMN].fillna('').tolist()
val_labels   = df_validation[LABEL_COLUMN].tolist()
test_texts   = df_test[TEXT_COLUMN].fillna('').tolist()
test_labels  = df_test[LABEL_COLUMN].tolist() if TEST_LABELS else None

train_dataset = IMDBDataset(train_texts, train_labels)
val_dataset   = IMDBDataset(val_texts,   val_labels)
test_dataset  = IMDBDataset(test_texts,  test_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
class SupervisedContrastiveLoss(nn.Module):
    """Supervised contrastive loss (Gunel et al. 2021).
    All math is done in fp32 to avoid overflow/nan."""
    def __init__(self, temperature=0.3):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings, labels):
        embeddings = embeddings.float()
        labels     = labels.long()
        B          = embeddings.shape[0]

        sim_matrix = torch.matmul(embeddings, embeddings.T) / self.temperature

        labels_col = labels.unsqueeze(1)
        mask       = torch.eq(labels_col, labels_col.T).float()

        self_mask  = torch.eye(B, device=embeddings.device).bool()
        mask       = mask.masked_fill(self_mask, 0)
        sim_matrix = sim_matrix.masked_fill(self_mask, float('-inf'))

        # Numerical stability: subtract max
        sim_max, _ = sim_matrix.max(dim=1, keepdim=True)
        sim_matrix = sim_matrix - sim_max.detach()

        exp_sim    = torch.exp(sim_matrix).masked_fill(self_mask, 0)
        denom      = exp_sim.sum(dim=1, keepdim=True) + 1e-8
        log_prob   = sim_matrix - torch.log(denom)

        pos_count  = mask.sum(dim=1)
        mean_lp    = (mask * log_prob).sum(dim=1) / (pos_count + 1e-8)

        valid      = pos_count > 0
        if valid.sum() == 0:
            return torch.tensor(0.0, device=embeddings.device, requires_grad=True)
        return -mean_lp[valid].mean()


class RoBERTaForSCL(nn.Module):
    def __init__(self, model_name, num_labels, proj_dim=128):
        super().__init__()
        self.roberta         = RobertaModel.from_pretrained(model_name)
        hidden               = self.roberta.config.hidden_size
        self.classifier      = nn.Sequential(nn.Dropout(0.1), nn.Linear(hidden, num_labels))
        self.projection_head = nn.Sequential(
            nn.Linear(hidden, proj_dim), nn.ReLU(), nn.Linear(proj_dim, proj_dim)
        )

    def forward(self, input_ids, attention_mask, return_projected=False):
        out     = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = out.last_hidden_state[:, 0, :]
        logits  = self.classifier(cls_emb)
        if return_projected:
            # CRITICAL: cast to fp32 BEFORE normalize to prevent fp16 overflow
            raw_proj = self.projection_head(cls_emb).float()
            proj     = F.normalize(raw_proj, p=2, dim=1)
            return logits, proj, cls_emb
        return logits, cls_emb

    def get_cls_embeddings(self, input_ids, attention_mask):
        with torch.no_grad():
            out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]

print('Model classes defined.')

In [ ]:
print('Loading CE model from zip ...')

if not os.path.exists(CE_ZIP_PATH):
    raise FileNotFoundError(
        f'{CE_ZIP_PATH} not found!\n'
        'Upload results_IMDB_RoBERTa-CE.zip to Colab before running.'
    )

ce_extract_dir = 'ce_model_extracted'
with zipfile.ZipFile(CE_ZIP_PATH, 'r') as zf:
    zf.extractall(ce_extract_dir)

model_ce = RoBERTaForSCL(MODEL_NAME, num_labels=2, proj_dim=PROJ_DIM).to(device)
model_ce.load_state_dict(torch.load(
    os.path.join(ce_extract_dir, 'model', 'model_ce.pt'),
    map_location=device
))
model_ce.eval()

hist_ce = {
    'train_losses': [0.3014, 0.1543, 0.0946],
    'val_losses':   [0.2358, 0.2711, 0.2493],
    'train_accs':   [0.8651, 0.9483, 0.9724],
    'val_accs':     [0.9104, 0.9162, 0.9336],
    'scl_losses':   [],
    'time':         1040.3,
}

print('CE model loaded. Previous Val Acc:', hist_ce['val_accs'][-1])

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Train SCL model — PURE FP32 (no autocast, no GradScaler)
#
# Why: autocast converts projection head output to fp16. F.normalize then
# computes L2 norm in fp16, which overflows (max fp16 = 65504) for 128-dim
# vectors, producing nan. The nan propagates into the contrastive loss.
# Disabling autocast entirely is the only 100% reliable fix.
# ══════════════════════════════════════════════════════════════════════════════
print('Training SCL model on IMDB (pure fp32) ...')
set_seed(SEED)

model_scl  = RoBERTaForSCL(MODEL_NAME, num_labels=2, proj_dim=PROJ_DIM).to(device)
scl_crit   = SupervisedContrastiveLoss(TEMPERATURE)
opt_scl    = torch.optim.AdamW(model_scl.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
sch_scl    = get_linear_schedule_with_warmup(opt_scl, warmup_steps, total_steps)

scl_train_losses_ce  = []
scl_val_losses_ce    = []
scl_train_losses_scl = []
scl_train_accs       = []
scl_val_accs         = []

start_scl = time.time()
for epoch in range(1, NUM_EPOCHS + 1):
    model_scl.train()
    ep_ce, ep_scl, preds_all, labs_all = 0.0, 0.0, [], []

    for batch in train_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['labels'].to(device)

        opt_scl.zero_grad()

        # NO autocast — everything runs in fp32
        logits, proj, _ = model_scl(ids, mask, return_projected=True)
        lce  = F.cross_entropy(logits, labs)
        lscl = scl_crit(proj, labs)
        loss = lce + LAMBDA_SCL * lscl

        # Sanity check on first batch of first epoch
        if epoch == 1 and ep_ce == 0.0:
            print(f'  [Sanity] proj range: [{proj.min().item():.4f}, {proj.max().item():.4f}]  '
                  f'lce={lce.item():.4f}  lscl={lscl.item():.4f}  '
                  f'any_nan={torch.isnan(proj).any().item()}')

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_scl.parameters(), 1.0)
        opt_scl.step()
        sch_scl.step()

        ep_ce  += lce.item()
        ep_scl += lscl.item()
        preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
        labs_all.extend(labs.cpu().numpy())

    model_scl.eval()
    vce, vp, vl_ = 0.0, [], []
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labs = batch['labels'].to(device)
            logits, _ = model_scl(ids, mask)
            vce += F.cross_entropy(logits, labs).item()
            vp.extend(torch.argmax(logits, 1).cpu().numpy())
            vl_.extend(labs.cpu().numpy())

    tce  = ep_ce  / len(train_loader)
    tscl = ep_scl / len(train_loader)
    vll  = vce    / len(val_loader)
    ta   = accuracy_score(labs_all, preds_all)
    va   = accuracy_score(vl_, vp)

    scl_train_losses_ce.append(tce)
    scl_val_losses_ce.append(vll)
    scl_train_losses_scl.append(tscl)
    scl_train_accs.append(ta)
    scl_val_accs.append(va)

    print(f'  [SCL] Epoch {epoch}/{NUM_EPOCHS} | CE: {tce:.4f} | SCL: {tscl:.4f} | '
          f'Val CE: {vll:.4f} | Tr Acc: {ta:.4f} | Val Acc: {va:.4f}')

train_time_scl = time.time() - start_scl

hist_scl = {
    'train_losses': scl_train_losses_ce,
    'val_losses':   scl_val_losses_ce,
    'train_accs':   scl_train_accs,
    'val_accs':     scl_val_accs,
    'scl_losses':   scl_train_losses_scl,
    'time':         train_time_scl,
}

print(f'SCL done in {train_time_scl:.1f}s  |  Final Val Acc: {scl_val_accs[-1]:.4f}')

In [ ]:
def evaluate_model(model, loader, has_labels=True):
    model.eval()
    preds, true_labs, total_loss = [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            logits, _ = model(ids, mask)
            if has_labels and 'labels' in batch:
                labs = batch['labels'].to(device)
                total_loss += F.cross_entropy(logits, labs).item()
                true_labs.extend(labs.cpu().numpy())
            preds.extend(torch.argmax(logits, 1).cpu().numpy())
    avg_loss = total_loss / len(loader) if true_labs else None
    return np.array(preds), np.array(true_labs) if true_labs else None, avg_loss

def compute_metrics(y_true, y_pred, loss=None):
    d = {
        'accuracy':            round(float(accuracy_score(y_true, y_pred)), 4),
        'precision_macro':     round(float(precision_score(y_true, y_pred, average='macro', zero_division=0)), 4),
        'recall_macro':        round(float(recall_score(y_true, y_pred, average='macro', zero_division=0)), 4),
        'f1_macro':            round(float(f1_score(y_true, y_pred, average='macro', zero_division=0)), 4),
        'precision_per_class': [round(float(x),4) for x in precision_score(y_true, y_pred, average=None, zero_division=0)],
        'recall_per_class':    [round(float(x),4) for x in recall_score(y_true, y_pred, average=None, zero_division=0)],
        'f1_per_class':        [round(float(x),4) for x in f1_score(y_true, y_pred, average=None, zero_division=0)],
        'confusion_matrix':    confusion_matrix(y_true, y_pred).tolist(),
    }
    if loss is not None:
        d['loss'] = round(float(loss), 4)
    return d

print('Evaluating CE ...')
y_tr_pred_ce,  y_tr_true_ce,  tr_loss_ce  = evaluate_model(model_ce, train_loader)
y_val_pred_ce, y_val_true_ce, val_loss_ce = evaluate_model(model_ce, val_loader)
y_test_pred_ce, y_test_true_ce, test_loss_ce = evaluate_model(model_ce, test_loader)

ce_train_res = compute_metrics(y_tr_true_ce,  y_tr_pred_ce,  tr_loss_ce)
ce_val_res   = compute_metrics(y_val_true_ce, y_val_pred_ce, val_loss_ce)
ce_test_res  = compute_metrics(y_test_true_ce, y_test_pred_ce, test_loss_ce)

print('Evaluating SCL ...')
y_tr_pred_scl,  y_tr_true_scl,  tr_loss_scl  = evaluate_model(model_scl, train_loader)
y_val_pred_scl, y_val_true_scl, val_loss_scl = evaluate_model(model_scl, val_loader)
y_test_pred_scl, y_test_true_scl, test_loss_scl = evaluate_model(model_scl, test_loader)

scl_train_res = compute_metrics(y_tr_true_scl,  y_tr_pred_scl,  tr_loss_scl)
scl_val_res   = compute_metrics(y_val_true_scl, y_val_pred_scl, val_loss_scl)
scl_test_res  = compute_metrics(y_test_true_scl, y_test_pred_scl, test_loss_scl)

class_labels = [int(x) for x in sorted(np.unique(y_val_true_ce))]

print()
print('=== RESULTS ===')
print(f'  CE  | Val: {ce_val_res["accuracy"]:.4f} | Test: {ce_test_res["accuracy"]:.4f} | F1: {ce_test_res["f1_macro"]:.4f}')
print(f'  SCL | Val: {scl_val_res["accuracy"]:.4f} | Test: {scl_test_res["accuracy"]:.4f} | F1: {scl_test_res["f1_macro"]:.4f}')
print()
print('Classification Report (CE - Test):')
print(classification_report(y_test_true_ce, y_test_pred_ce, zero_division=0))
print('Classification Report (SCL - Test):')
print(classification_report(y_test_true_scl, y_test_pred_scl, zero_division=0))

In [ ]:
epochs_range = list(range(1, NUM_EPOCHS + 1))

def make_plots(hist, val_res, label, test_res=None, cmap='Blues'):
    er = list(range(1, len(hist['train_accs']) + 1))
    cm_data = test_res['confusion_matrix'] if test_res else val_res['confusion_matrix']
    cm_tag  = '(test)' if test_res else '(val)'

    fig_cm, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(np.array(cm_data), annot=True, fmt='d', cmap=cmap,
                xticklabels=class_labels, yticklabels=class_labels, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'{label} - IMDB {cm_tag}', fontweight='bold')
    fig_cm.tight_layout()

    fig_acc, ax = plt.subplots(figsize=(9, 5))
    ax.plot(er, hist['train_accs'], label='Train', linewidth=2, color='#1f77b4')
    ax.plot(er, hist['val_accs'],   label='Val',   linewidth=2, color='#ff7f0e')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.set_title(f'{label} - IMDB', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3); fig_acc.tight_layout()

    fig_loss, ax = plt.subplots(figsize=(9, 5))
    ax.plot(er, hist['train_losses'], label='Train CE', linewidth=2, color='#1f77b4')
    ax.plot(er, hist['val_losses'],   label='Val CE',   linewidth=2, color='#ff7f0e')
    if hist['scl_losses']:
        ax.plot(er, hist['scl_losses'], label='Train SCL', linewidth=2, color='#2ca02c', linestyle='--')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title(f'{label} - IMDB', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3); fig_loss.tight_layout()

    return {'confusion_matrix.png': fig_cm, 'accuracy_curves.png': fig_acc, 'loss_curves.png': fig_loss}

figs_ce  = make_plots(hist_ce,  ce_val_res,  'RoBERTa-CE',  ce_test_res, 'Blues')
figs_scl = make_plots(hist_scl, scl_val_res, 'RoBERTa-SCL', scl_test_res, 'Greens')

fig_compare, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, hist_ce['val_accs'],  label='CE Val Acc',  linewidth=2, color='#1f77b4')
ax.plot(epochs_range, hist_scl['val_accs'], label='SCL Val Acc', linewidth=2, color='#d62728')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('CE vs SCL - Validation Accuracy - IMDB', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); fig_compare.tight_layout()
figs_ce['ce_vs_scl_compare.png']  = fig_compare
figs_scl['ce_vs_scl_compare.png'] = fig_compare

print('All figures created.')

In [ ]:
def save_results(model_obj, figures,
                 train_results, val_results, test_results,
                 train_time, entries_per_second,
                 dataset, model_name, model_hf_name,
                 num_epochs, batch_size, learning_rate, max_length,
                 text_column, label_column, class_labels,
                 df_train, df_validation, df_test, root_dir,
                 extra_params=None, optimizer_info=None, scheduler_info=None):

    model_size_bytes = sum(p.numel() * p.element_size() for p in model_obj.parameters())
    ts       = datetime.now().strftime('%d_%m_%Y_%H_%M_%S')
    run_name = f'{model_hf_name}_ep{num_epochs}_bs{batch_size}_lr{learning_rate}_{ts}'
    out_dir  = os.path.join(root_dir, model_name, dataset, run_name)
    os.makedirs(out_dir, exist_ok=True)

    for fig_name, fig_obj in figures.items():
        fig_obj.savefig(os.path.join(out_dir, fig_name), dpi=150, bbox_inches='tight')

    total_p     = sum(p.numel() for p in model_obj.parameters())
    trainable_p = sum(p.numel() for p in model_obj.parameters() if p.requires_grad)

    metrics = {
        'model': model_name, 'model_hf_name': model_hf_name,
        'dataset': dataset, 'run_name': run_name, 'timestamp': ts, 'seed': SEED,
        'model_params': {
            'num_epochs': num_epochs, 'batch_size': batch_size,
            'learning_rate': learning_rate, 'max_length': max_length,
            'total_params': total_p, 'trainable_params': trainable_p,
            **(extra_params or {}),
        },
        'model_size': {'bytes': model_size_bytes, 'MB': round(model_size_bytes/(1024*1024), 4)},
        'training': {
            'training_time_seconds': round(train_time, 4),
            'entries_per_second':    round(entries_per_second, 2),
            'num_epochs': num_epochs, 'train_samples': len(df_train),
        },
        'data': {
            'train_samples': len(df_train), 'validation_samples': len(df_validation),
            'test_samples': len(df_test) if df_test is not None else 0,
            'text_column': text_column, 'label_column': label_column,
            'classes': list(class_labels),
        },
        'optimizer': optimizer_info or {}, 'scheduler': scheduler_info or {},
        'train_results': train_results, 'validation_results': val_results,
    }
    if test_results is not None:
        metrics['test_results'] = test_results

    mp = os.path.join(out_dir, 'metrics.json')
    with open(mp, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f'Saved: {out_dir}')
    return out_dir, mp, run_name

opt_info = {'name': 'AdamW', 'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY}
sch_info = {'name': 'linear_warmup', 'warmup_ratio': WARMUP_RATIO, 'total_steps': total_steps}

output_dir_ce, _, _ = save_results(
    model_obj=model_ce, figures=figs_ce,
    train_results=ce_train_res, val_results=ce_val_res, test_results=ce_test_res,
    train_time=hist_ce['time'],
    entries_per_second=(len(train_labels)*NUM_EPOCHS)/hist_ce['time'],
    dataset=DATASET, model_name='RoBERTa-CE', model_hf_name=MODEL_NAME,
    num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, max_length=MAX_LENGTH,
    text_column=TEXT_COLUMN, label_column=LABEL_COLUMN, class_labels=class_labels,
    df_train=df_train, df_validation=df_validation, df_test=df_test, root_dir=ROOT_DIR,
    extra_params={'lambda_scl': 0.0},
    optimizer_info=opt_info, scheduler_info=sch_info,
)

output_dir_scl, _, _ = save_results(
    model_obj=model_scl, figures=figs_scl,
    train_results=scl_train_res, val_results=scl_val_res, test_results=scl_test_res,
    train_time=hist_scl['time'],
    entries_per_second=(len(train_labels)*NUM_EPOCHS)/hist_scl['time'],
    dataset=DATASET, model_name='RoBERTa-SCL', model_hf_name=MODEL_NAME,
    num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, max_length=MAX_LENGTH,
    text_column=TEXT_COLUMN, label_column=LABEL_COLUMN, class_labels=class_labels,
    df_train=df_train, df_validation=df_validation, df_test=df_test, root_dir=ROOT_DIR,
    extra_params={'temperature': TEMPERATURE, 'lambda_scl': LAMBDA_SCL, 'proj_dim': PROJ_DIM},
    optimizer_info=opt_info, scheduler_info=sch_info,
)

print()
print('=== SAVED ===')
print(f'  CE:  {output_dir_ce}')
print(f'  SCL: {output_dir_scl}')

In [ ]:
print('Extracting CLS embeddings for t-SNE ...')

def get_cls_embs(model, loader):
    model.eval()
    embs, labs = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            e    = model.get_cls_embeddings(ids, mask)
            embs.append(e.cpu().numpy())
            if 'labels' in batch:
                labs.extend(batch['labels'].numpy())
    return np.vstack(embs), np.array(labs)

emb_ce,  lab_ce  = get_cls_embs(model_ce,  val_loader)
emb_scl, lab_scl = get_cls_embs(model_scl, val_loader)

print('Running t-SNE ...')
tsne_model = TSNE(n_components=2, perplexity=30, random_state=SEED, max_iter=1000)
coords_ce  = tsne_model.fit_transform(emb_ce)
coords_scl = tsne_model.fit_transform(emb_scl)

colors = {0: '#d62728', 1: '#1f77b4'}
fig_tsne, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, coords, labs, title in [
        (axes[0], coords_ce,  lab_ce,  'RoBERTa-CE'),
        (axes[1], coords_scl, lab_scl, 'RoBERTa-SCL'),
]:
    for cls_id, color in colors.items():
        m = labs == cls_id
        ax.scatter(coords[m, 0], coords[m, 1], c=color, alpha=0.6, s=10,
                   label='Positive' if cls_id == 1 else 'Negative')
    ax.set_title(f'{title} - t-SNE (IMDB val)', fontweight='bold')
    ax.legend(markerscale=3); ax.axis('off')
fig_tsne.tight_layout()
fig_tsne.savefig(os.path.join(output_dir_ce,  'tsne.png'), dpi=150, bbox_inches='tight')
fig_tsne.savefig(os.path.join(output_dir_scl, 'tsne.png'), dpi=150, bbox_inches='tight')
plt.show()
print('t-SNE saved.')

In [ ]:
for out_dir, model, name in [
    (output_dir_ce,  model_ce,  'model_ce.pt'),
    (output_dir_scl, model_scl, 'model_scl.pt'),
]:
    mdir = os.path.join(out_dir, 'model')
    os.makedirs(mdir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(mdir, name))
    tokenizer.save_pretrained(mdir)
    size_mb = os.path.getsize(os.path.join(mdir, name)) / (1024*1024)
    print(f'  {name}: {size_mb:.0f} MB -> {mdir}')

In [ ]:
zip_ce  = shutil.make_archive('results_IMDB_RoBERTa-CE',  'zip', output_dir_ce)
zip_scl = shutil.make_archive('results_IMDB_RoBERTa-SCL', 'zip', output_dir_scl)

print('Zip files:')
for z in [zip_ce, zip_scl]:
    size_mb = os.path.getsize(z) / (1024 * 1024)
    print(f'  {os.path.basename(z)}  ({size_mb:.1f} MB)')

try:
    from google.colab import files
    files.download(zip_ce)
    files.download(zip_scl)
except ImportError:
    print('Not in Colab - download manually.')